# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstruct_hdf5_from_s3.py` but uses the **local filesystem** for zarr storage and **SQLite** for metadata. No cloud credentials needed.

**Workflow**
1. Ingest a GMI granule → zarr groups on local disk + SQLite metadata
2. Find intersecting data for a bounding box via STARE SIDs + SQLite
3. Load intersecting zarr chunks from disk
4. Reconstruct an HDF5 file (both S1 and S2 scans)
5. Compare the reconstructed structure with the original granule
6. Verify SQLite metadata

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
                       "-q"])

In [ ]:
import os
import sqlite3
import h5py
from starepandas.demo import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [ ]:
# zarr store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

GRANULE_FILE = (
    "/Users/tonhai/workspace/Bayesics/L1C_Data_Samples/GPM/2025/Jan_1_2/"
    "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5"
)

# Bounding box filter — set to None to reconstruct the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_local_reconstructed.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstructed HDF5 (e.g. 3× the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

## Step 1 — Ingest granule → local zarr + SQLite

In [ ]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

In [ ]:
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=10)
print(f"Written {len(local_paths)} scan path(s).")
for p in local_paths:
    print(f"  {p}")

## Step 2 — Find intersecting data via STARE SIDs

In [ ]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=10)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all groups will be loaded (full granule reconstruction)")

intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'])
print(f"Found {len(intersecting)} metadata row(s).")
intersecting[['Dataset', 'grouped_id', 'group_path']]

## Step 3 — Load intersecting zarr chunks from disk

In [ ]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting chunks found.")
    data_dict = {}

## Step 4 — Reconstruct HDF5 (S1 + S2)

In [ ]:
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_local_prefix = os.path.join(LOCAL_ROOT, granule_basename)

recon_path = demo.reconstruct_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    local_prefix=granule_local_prefix,
)
print(f"Written to: {recon_path}")

## Step 5 — Structure comparison: reconstructed vs original

In [ ]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTRUCTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")

## Step 6 — SQLite metadata verification

In [ ]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} group(s)")